In [13]:
import numpy as np, pandas as pd, pickle
from pathlib import Path
from scipy.sparse import load_npz, csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.special import logsumexp
from tqdm.auto import tqdm

BASE_DIR = Path('/Users/gre_en/Documents/Analysis/Projects/1_research/Sci-Soc')
NETWORKS_DIR = BASE_DIR / "data" / "processed" / "networks"
BACKBONE_DIR = BASE_DIR / "data" / "processed" / "backbone"
EMB_DIR = BASE_DIR / "data" / "processed" / "embeddings"
SOURCES = ["news", "paper"]

### Backbone

In [14]:
def load_node_info(source, net_dir):
    with open(Path(net_dir) / source / "node_info.pkl", "rb") as f:
        return pickle.load(f)


def disparity_alpha(W):
    """
    Serrano et al. (2009) in closed form: alpha_ij = (1 - w_ij/s_i)^(k_i - 1).
    """
    W = W.tocsr()
    s = np.asarray(W.sum(axis=1)).ravel()
    k = np.diff(W.indptr)

    coo = W.tocoo()
    keep = coo.row < coo.col
    i, j, w = coo.row[keep], coo.col[keep], coo.data[keep]

    def alpha(node, w_):
        kk = k[node]
        p = np.divide(w_, s[node], out=np.zeros_like(w_), where=s[node] > 0)
        a = np.power(1.0 - p, np.maximum(kk - 1, 0))
        return np.where(kk > 1, a, 1.0)

    a_i, a_j = alpha(i, w), alpha(j, w)
    return pd.DataFrame({"i": i, "j": j, "w": w,
                         "alpha_min": np.minimum(a_i, a_j)})


def rca_score(W):
    W = W.tocsr()
    s = np.asarray(W.sum(axis=1)).ravel()
    total = s.sum() / 2.0

    coo = W.tocoo()
    keep = coo.row < coo.col
    i, j, w = coo.row[keep], coo.col[keep], coo.data[keep]

    denom = s[i] * s[j]
    rca = np.divide(w * total, denom, out=np.zeros_like(w), where=denom > 0)
    return pd.DataFrame({"i": i, "j": j, "w": w, "rca": rca})


def extract_backbone(W, density_k=3.0, method="disparity"):
    if method == "disparity":
        df = disparity_alpha(W)
        sort_cols, asc = ["alpha_min", "w"], [True, False]
    elif method == "rca":
        df = rca_score(W)
        sort_cols, asc = ["rca", "w"], [False, False]
    else:
        raise ValueError(method)

    if df.empty:
        return df.assign(mst_only=pd.Series(dtype=int))

    n_active = int((np.asarray(W.sum(axis=1)).ravel() > 0).sum())
    n_keep = min(int(round(density_k * n_active)), len(df))

    df = df.sort_values(sort_cols, ascending=asc, kind="mergesort").reset_index(drop=True)
    cut = df.iloc[:n_keep].copy()
    cut["mst_only"] = 0

    neg = csr_matrix((-W.data, W.indices, W.indptr), shape=W.shape)
    mst = minimum_spanning_tree(neg).tocoo()
    mst_pairs = {(min(a, b), max(a, b)) for a, b in zip(mst.row, mst.col)}
    have = set(zip(cut.i.to_numpy(), cut.j.to_numpy()))
    add = mst_pairs - have

    if add:
        rest = df.iloc[n_keep:]
        idx = pd.MultiIndex.from_arrays([rest.i, rest.j])
        extra = rest[idx.isin(list(add))].copy()
        extra["mst_only"] = 1
        cut = pd.concat([cut, extra], ignore_index=True)
    return cut


def build_backbones(source, info, net_dir, out_dir, density_k=3.0):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for year in tqdm(info["years"], desc=source):
        W = load_npz(Path(net_dir) / source / f"adj_{year}.npz").tocsr()
        bb = extract_backbone(W, density_k)
        bb.to_parquet(out_dir / f"{source}_{year}_backbone.parquet")
        nodes = np.unique(np.concatenate([bb.i.to_numpy(), bb.j.to_numpy()]))
        rows.append({"year": int(year), "E_bb": len(bb),
                     "mst_share": float((bb.mst_only == 1).mean()),
                     "mean_degree": 2 * len(bb) / max(len(nodes), 1),
                     "implied_alpha": float(bb.loc[bb.mst_only == 0, "alpha_min"].max())})
    s = pd.DataFrame(rows)
    s.to_csv(out_dir / f"{source}_backbone_summary.csv", index=False)
    return s

In [ ]:
info = {s: load_node_info(s, NETWORKS_DIR) for s in SOURCES}

for s in SOURCES:
    print(build_backbones(s, info[s], NETWORKS_DIR, BACKBONE_DIR).to_string(index=False))

In [ ]:
source = "news"
info_s = load_node_info(source, NETWORKS_DIR)

rows = []
for year in tqdm(info_s["years"], desc=source):
    W = load_npz(NETWORKS_DIR / source / f"adj_{year}.npz").tocsr()
    n_active = int((np.asarray(W.sum(axis=1)).ravel() > 0).sum())
    deg = np.diff(W.indptr).astype(float)

    bb_d = extract_backbone(W, 3.0, "disparity")
    bb_r = extract_backbone(W, 3.0, "rca")

    set_d = set(zip(bb_d.i.to_numpy(), bb_d.j.to_numpy()))
    set_r = set(zip(bb_r.i.to_numpy(), bb_r.j.to_numpy()))

    r = {"year": int(year), "N_t": n_active, "E_t": W.nnz // 2,
         "jaccard": len(set_d & set_r) / len(set_d | set_r)}

    for m, d in [("disparity", bb_d), ("rca", bb_r)]:
        nodes = np.unique(np.concatenate([d.i.to_numpy(), d.j.to_numpy()]))
        r[f"{m}_cov"] = len(nodes) / n_active
        r[f"{m}_wret"] = float(d.w.sum() / (W.data.sum() / 2))
        r[f"{m}_deg"] = float(deg[nodes].mean())   # original degree of surviving nodes

    # fixed alpha = 0.05, the sensitivity item
    da = disparity_alpha(W)
    sig = da[da.alpha_min < 0.05]
    n_sig = len(np.unique(np.concatenate([sig.i.to_numpy(), sig.j.to_numpy()]))) if len(sig) else 1
    r["a05_E"] = len(sig)
    r["a05_mean_degree"] = 2 * len(sig) / n_sig
    rows.append(r)

cmp_news = pd.DataFrame(rows)
print(cmp_news.to_string(index=False))

### Persistent path

In [15]:
def persistent_edges(source, info, bb_dir, min_years=20):
    """
    Edges present in the backbone for at least min_years consecutive years.
    """
    years = list(info["years"])
    present = {}
    for t, year in enumerate(years):
        bb = pd.read_parquet(Path(bb_dir) / f"{source}_{year}_backbone.parquet")
        for i, j in zip(bb.i.to_numpy(), bb.j.to_numpy()):
            present.setdefault((i, j), np.zeros(len(years), bool))[t] = True

    out = []
    for (i, j), mask in present.items():
        run = best = 0
        for v in mask:
            run = run + 1 if v else 0
            best = max(best, run)
        if best >= min_years:
            out.append((i, j, best))
    return pd.DataFrame(out, columns=["i", "j", "run"])


def persistent_paths(pers_edges, length=4, max_paths=200, seed=0):
    """
    One path per randomly drawn start node.
    """
    rng = np.random.default_rng(seed)
    adj = {}
    for i, j in zip(pers_edges.i, pers_edges.j):
        adj.setdefault(i, set()).add(j)
        adj.setdefault(j, set()).add(i)

    paths, seen = [], set()
    for s in rng.permutation(list(adj)):
        path = [int(s)]
        while len(path) < length:
            cand = list(adj[path[-1]] - set(path))
            if not cand:
                break
            path.append(int(rng.choice(cand)))
        if len(path) < length:
            continue
        key = tuple(sorted(path))
        if key in seen:
            continue
        seen.add(key)
        paths.append(path)
        if len(paths) >= max_paths:
            break
    return paths

In [17]:
pe = {s: persistent_edges(s, info[s], BACKBONE_DIR, min_years=20) for s in SOURCES}
paths = {s: persistent_paths(pe[s]) for s in SOURCES}
print({s: (len(pe[s]), len(paths[s])) for s in SOURCES})

for s in SOURCES:
    nodes = pd.Series([n for p in paths[s] for n in p])
    print(s, "고유 노드", nodes.nunique(), "/ 총", len(nodes))
    
    nodes = sorted({n for p in paths[s] for n in p})
    sub = pe[s][pe[s].i.isin(nodes) & pe[s].j.isin(nodes)]
    n = len(nodes)
    print(s, "노드", n, "지속엣지", len(sub),
          "밀도", round(2*len(sub)/(n*(n-1)), 3))

with open(BACKBONE_DIR / "persistent_paths.pkl", "wb") as f:
    pickle.dump({"paths": paths, "pers_edges": pe, "min_years": 20}, f)

news:   0%|          | 0/34 [00:00<?, ?it/s]

 year  E_bb  mst_share  mean_degree  implied_alpha
 1990  8450   0.135858     6.943303       0.199992
 1991  5736   0.129707     6.894231       0.377150
 1992  8330   0.138535     6.964883       0.218998
 1993  8440   0.133768     6.926549       0.233469
 1994  9135   0.132348     6.915216       0.198718
 1995  9207   0.133920     6.927765       0.231982
 1996  9211   0.133645     6.925564       0.276151
 1997  9631   0.132489     6.916338       0.222149
 1998 15330   0.140313     6.979285       0.096988
 1999 16275   0.141014     6.984979       0.091125
 2000 16708   0.142985     7.001048       0.083513
 2001 16380   0.141758     6.991037       0.087952
 2002 16765   0.141426     6.988328       0.100880
 2003 16937   0.134557     6.932869       0.109557
 2004 18098   0.139518     6.972838       0.093345
 2005 18587   0.139721     6.974484       0.093182
 2006 18672   0.136247     6.946429       0.094972
 2007 21416   0.142137     6.994121       0.056902
 2008 22485   0.147698     7.03

paper:   0%|          | 0/34 [00:00<?, ?it/s]

 year  E_bb  mst_share  mean_degree  implied_alpha
 1990 43867   0.138783     6.966886       0.096818
 1991 44112   0.139214     6.970372       0.088931
 1992 45386   0.141630     6.989989       0.080515
 1993 45685   0.141140     6.986008       0.076251
 1994 46162   0.141697     6.990535       0.070262
 1995 47664   0.142938     7.000661       0.069471
 1996 48254   0.143656     7.006534       0.062207
 1997 48942   0.142454     6.996712       0.060831
 1998 49307   0.143083     7.001846       0.055914
 1999 50744   0.144825     7.016108       0.049387
 2000 52547   0.143624     7.006267       0.041988
 2001 53454   0.144012     7.009441       0.037308
 2002 56273   0.156452     7.112810       0.010825
 2003 56874   0.152653     7.080926       0.015860
 2004 57353   0.151152     7.068400       0.015303
 2005 58326   0.152916     7.083126       0.010776
 2006 59501   0.154014     7.092318       0.008272
 2007 60480   0.153819     7.090685       0.007546
 2008 61135   0.154543     7.09

### Perplexity

In [16]:
def path_pseudo_perplexity(paths, Z_t, active_idx, tau=1.0, normalize=True):
    pos = {int(n): k for k, n in enumerate(active_idx)}
    Za = Z_t[active_idx]
    if normalize:
        Za = Za / (np.linalg.norm(Za, axis=1, keepdims=True) + 1e-12)

    rows = []
    for p in paths:
        local = [pos[v] for v in p if v in pos]
        if len(local) < 2:
            continue

        logps = []
        for k in range(len(local)):
            ctx = np.delete(np.asarray(local), k)
            c = Za[ctx].mean(axis=0)
            s = Za @ c / tau
            s[local[k]] = s[local[k]]          # target stays in the candidate set
            logps.append(s[local[k]] - logsumexp(s))

        rows.append({"path": tuple(p),
                     "n_scored": len(local),
                     "ppl": float(np.exp(-np.mean(logps))),
                     "n_active": len(active_idx)})
    return pd.DataFrame(rows)


def run_pseudo_perplexity(source, info, paths, emb_path, keep_path, tau=1.0):
    """
    One row per (path, year).
    """
    Z = np.load(emb_path)                                  # [N_local, T, F]
    keep = np.load(keep_path)
    g2l = {int(g): i for i, g in enumerate(keep)}
    paths_l = [[g2l[v] for v in p if v in g2l] for p in paths]
    paths_l = [p for p in paths_l if len(p) >= 2]

    freq = np.asarray(info["concept_freq_year"])[:, keep]  # [T, N_local]
    out = []
    for t, year in enumerate(tqdm(info["years"], desc=source)):
        active = np.where(freq[t] > 0)[0]
        d = path_pseudo_perplexity(paths_l, Z[:, t, :], active, tau)
        d["source"], d["year"] = source, int(year)
        out.append(d)

    d = pd.concat(out, ignore_index=True)
    d["ppl_rel"] = d["ppl"] / d["n_active"]
    return d

In [ ]:
# pp = run_pseudo_perplexity(
#     "news", info["news"], paths["news"],
#     EMB_DIR / "news_embs_all.npy",
#     BASE_DIR / "data" / "processed" / "dysat_input" / "news" / "keep.npy")

# print(pp.groupby("year")[["ppl", "ppl_rel", "n_active"]].median().round(4))

### (old) test

In [18]:
with open(BACKBONE_DIR / "persistent_paths.pkl", "rb") as f:
    saved = pickle.load(f)

paths = saved["paths"]
pe = saved["pers_edges"]

for s in SOURCES:
    print(s, "paths", len(paths[s]), "| persistent edges", len(pe[s]),
          "| unique nodes", len({n for p in paths[s] for n in p}))
    
info = {s: load_node_info(s, NETWORKS_DIR) for s in SOURCES}

news paths 200 | persistent edges 2235 | unique nodes 363
paper paths 200 | persistent edges 23364 | unique nodes 535


In [19]:
import torch, numpy as np

Z_old = torch.load(BASE_DIR / "data" / "processed_old" / "embeddings" / "news_embeddings.pt",
                   map_location="cpu")
print(type(Z_old))
if hasattr(Z_old, "shape"):
    print(Z_old.shape)
else:
    print(Z_old.keys() if hasattr(Z_old, "keys") else Z_old)


print("현재 path 최대 인덱스:", max(max(p) for p in paths["news"]))
print("현재 전역 어휘:", len(info["news"]["vocab"]))

<class 'torch.Tensor'>
torch.Size([34, 29312, 32])
현재 path 최대 인덱스: 34160
현재 전역 어휘: 34403


In [20]:
def run_pseudo_perplexity_old(source, info, paths, emb_path, tau=1.0):
    """
    Same scoring, reading the older .pt tensors: they are [T, N, F] and indexed
    by the previous vocabulary, so there is no keep.npy to map through.
    """
    Z = torch.load(emb_path, map_location="cpu").numpy()   # [T, N, F]
    Z = Z.transpose(1, 0, 2)                               # [N, T, F]
    N = Z.shape[0]

    paths_l = [[v for v in p if v < N] for p in paths]
    paths_l = [p for p in paths_l if len(p) >= 2]

    freq = np.asarray(info["concept_freq_year"])[:, :N]
    out = []
    for t, year in enumerate(tqdm(info["years"], desc=source)):
        active = np.where(freq[t] > 0)[0]
        d = path_pseudo_perplexity(paths_l, Z[:, t, :], active, tau)
        d["source"], d["year"] = source, int(year)
        out.append(d)

    d = pd.concat(out, ignore_index=True)
    d["ppl_rel"] = d["ppl"] / d["n_active"]
    return d

In [21]:
pp = run_pseudo_perplexity_old(
    "news", info["news"], paths["news"],
    BASE_DIR / "data" / "processed_old" / "embeddings" / "news_embeddings.pt")

print(pp.groupby("year")[["ppl", "ppl_rel", "n_active"]].median().round(4))

news:   0%|          | 0/34 [00:00<?, ?it/s]

            ppl  ppl_rel  n_active
year                              
1990  2466.3905   1.1829    2085.0
1991  1578.7726   1.0987    1437.0
1992  2386.3958   1.1590    2059.0
1993  2358.1365   1.1343    2079.0
1994  2585.6096   1.1301    2288.0
1995  2627.0999   1.1363    2312.0
1996  2647.3699   1.1525    2297.0
1997  2796.6545   1.1590    2413.0
1998  4613.2590   1.2159    3794.0
1999  4918.3162   1.2244    4017.0
2000  5026.8303   1.2163    4133.0
2001  4923.0718   1.2243    4021.0
2002  5146.2239   1.2418    4144.0
2003  5225.5789   1.2427    4205.0
2004  5574.2319   1.2481    4466.0
2005  5676.9526   1.2409    4575.0
2006  5739.6626   1.2432    4617.0
2007  6563.4910   1.2426    5282.0
2008  6865.0793   1.2457    5511.0
2009  6551.1536   1.2363    5299.0
2010  6108.8845   1.2419    4919.0
2011  6015.8801   1.2401    4851.0
2012  6282.6169   1.2475    5036.0
2013  6333.1804   1.2226    5180.0
2014  6487.0762   1.2437    5216.0
2015  6464.6746   1.2466    5186.0
2016  6957.0767   1.